In [ ]:
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok nltk streamlit-option-menu plotly textstat PyPDF2 -q
!pip install requests beautifulsoup4 tqdm urllib3 sentence-transformers faiss-cpu pypdf -q
!pip install deep-translator
!pip install pyvis -q
!python -m spacy download en_core_web_sm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/

In [ ]:
%%writefile db.py
import sqlite3
import bcrypt
import datetime
import time

DB_NAME = "users.db"
max_login_attempts = 3
lockout_time = 300  # 5 minutes in seconds

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    # Users table
    c.execute('''CREATE TABLE IF NOT EXISTS users
                 (email TEXT PRIMARY KEY, password BLOB, created_at TEXT)''')

    # Password History table
    c.execute('''CREATE TABLE IF NOT EXISTS password_history
                 (id INTEGER PRIMARY KEY AUTOINCREMENT,
                  email TEXT,
                  password BLOB,
                  set_at TEXT,
                  FOREIGN KEY(email) REFERENCES users(email))''')

    # Login Attempts table
    c.execute('''CREATE TABLE IF NOT EXISTS login_attempts
                 (email TEXT PRIMARY KEY, attempts INTEGER, last_attempt REAL)''')

    conn.commit()
    conn.close()

    # Ensure Admin Exists
    init_admin()

def init_admin():
    if not check_user_exists("admin@llm.com"):
        register_user("admin@llm.com", "Admin@123")
        print("Admin account created.")

def _get_timestamp():
    return datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

# --- Rate Limiting ---
def get_login_attempts(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    return data if data else (0, 0)

def increment_login_attempts(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    attempts, _ = get_login_attempts(email)
    new_attempts = attempts + 1
    now = time.time()

    c.execute("INSERT OR REPLACE INTO login_attempts (email, attempts, last_attempt) VALUES (?, ?, ?)",
              (email, new_attempts, now))
    conn.commit()
    conn.close()

def reset_login_attempts(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM login_attempts WHERE email = ?", (email,))
    conn.commit()
    conn.close()

def is_rate_limited(email):
    attempts, last_attempt = get_login_attempts(email)
    if attempts >= max_login_attempts:
        if time.time() - last_attempt < lockout_time:
            return True, lockout_time - (time.time() - last_attempt)
        else:
            # Lockout expired
            reset_login_attempts(email)
    return False, 0

# --- User Management ---
def register_user(email, password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    try:
        salt = bcrypt.gensalt()
        hashed = bcrypt.hashpw(password.encode('utf-8'), salt)
        now = _get_timestamp()

        c.execute("INSERT INTO users (email, password, created_at) VALUES (?, ?, ?)", (email, hashed, now))
        c.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)", (email, hashed, now))

        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

def authenticate_user(email, password):
    # Check rate limit first in app.py or here? Better to check before auth logic.
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password FROM users WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    if data:
        stored_hash = data[0]
        if bcrypt.checkpw(password.encode('utf-8'), stored_hash):
            reset_login_attempts(email) # Success reset
            return True

    # Failed
    increment_login_attempts(email)
    return False

def check_is_old_password(email, password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password, set_at FROM password_history WHERE email = ? ORDER BY set_at DESC", (email,))
    history = c.fetchall()
    conn.close()

    for stored_hash, set_at in history:
        if bcrypt.checkpw(password.encode('utf-8'), stored_hash):
            return set_at
    return None

def check_password_reused(email, new_password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password FROM password_history WHERE email = ?", (email,))
    history = c.fetchall()
    conn.close()

    for (stored_hash,) in history:
        if bcrypt.checkpw(new_password.encode('utf-8'), stored_hash):
            return True
    return False

def check_user_exists(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT 1 FROM users WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    return data is not None

def update_password(email, new_password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    salt = bcrypt.gensalt()
    hashed = bcrypt.hashpw(new_password.encode('utf-8'), salt)
    now = _get_timestamp()

    c.execute("UPDATE users SET password = ? WHERE email = ?", (hashed, email))
    c.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)", (email, hashed, now))

    conn.commit()
    conn.close()

# --- Admin Functions ---
def get_all_users():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT email, created_at FROM users")
    data = c.fetchall()
    conn.close()
    return data

def delete_user(email):
    if email == "admin@llm.com": return False # Protect admin
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM users WHERE email = ?", (email,))
    c.execute("DELETE FROM password_history WHERE email = ?", (email,))
    c.execute("DELETE FROM login_attempts WHERE email = ?", (email,))
    conn.commit()
    conn.close()
    return True


Writing db.py


In [ ]:
%%writefile readability.py
import textstat

class ReadabilityAnalyzer:
    def __init__(self, text):
        self.text = text
        self.num_sentences = textstat.sentence_count(text)
        self.num_words = textstat.lexicon_count(text, removepunct=True)
        self.num_syllables = textstat.syllable_count(text)
        self.complex_words = textstat.difficult_words(text)
        self.char_count = textstat.char_count(text)

    def get_all_metrics(self):
        return {
            "Flesch Reading Ease": textstat.flesch_reading_ease(self.text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(self.text),
            "SMOG Index": textstat.smog_index(self.text),
            "Gunning Fog": textstat.gunning_fog(self.text),
            "Coleman-Liau": textstat.coleman_liau_index(self.text)
        }

Writing readability.py


In [ ]:
%%writefile knowledge_graph.py
import spacy, networkx as nx, os
from pyvis.network import Network

# Attempt to load spaCy for Entity Extraction
try:
    nlp = spacy.load("en_core_web_sm")
except:
    # If not installed, run: !python -m spacy download en_core_web_sm
    nlp = None

class PolicyKG:
    def __init__(self):
        self.G = nx.Graph()

    def add_policy_data(self, chunks):
        if not nlp:
            return

        # Mapping relationships for the first 40 chunks to ensure clarity
        max_chunks = min(len(chunks), 40)

        for i in range(max_chunks):
            text = chunks[i]['text'][:1000] # Relationship mapping from text content
            source_file = chunks[i]['source']
            doc = nlp(text)

            # Entity Extraction (ORG=Organization, GPE=Location, LAW=Policies/Laws)
            entities = [ent.text.strip() for ent in doc.ents
                        if ent.label_ in ['ORG', 'GPE', 'LAW', 'PERSON']
                        and len(ent.text.strip()) > 2]

            source_node = f"Doc: {source_file}"
            self.G.add_node(source_node, title="Policy Document", color="#764ba2", size=25)

            for ent in entities:
                # Relationship Mapping: Connecting Document to Extracted Entities
                self.G.add_node(ent, title="Entity", color="#17a2b8", size=20)
                self.G.add_edge(source_node, ent)

    def visualize_kg(self):
        # Interactive Visualization using Pyvis
        net = Network(height="650px", width="100%", bgcolor="#000000", font_color="white", directed=False)
        net.from_nx(self.G)

        # Keep the graph stable (No constant moving)
        net.set_options("""
        var options = {
          "physics": {
            "barnesHut": {
              "gravitationalConstant": -30000,
              "centralGravity": 0.3,
              "springLength": 200
            },
            "stabilization": {
              "enabled": true,
              "iterations": 1000
            }
          }
        }
        """)

        path = "policy_kg.html"
        net.save_graph(path)
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()

Writing knowledge_graph.py


In [ ]:
%%writefile app.py
import streamlit as st
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import secrets
import bcrypt
import jwt
import datetime
import time
import os
import re
import hmac
import hashlib
import struct
import db
from streamlit_option_menu import option_menu
import plotly.graph_objects as go
import PyPDF2
import pickle
import streamlit.components.v1 as components
from knowledge_graph import PolicyKG

# --- NEW IMPORTS FOR YOUR SUMMARIZATION ---
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from deep_translator import GoogleTranslator

# --- Configuration ---
EMAIL_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")
SECRET_KEY = os.getenv("JWT_SECRET_KEY")
EMAIL_ADDRESS = "bhuvireddy2706@gmail.com"
OTP_EXPIRY_MINUTES = 10

# --- Database Initialization ---
if 'db_initialized' not in st.session_state:
    db.init_db()
    st.session_state['db_initialized'] = True

# --- UI Theme (Neon Style) ---
st.set_page_config(page_title="PolicyNav Login", layout="wide")

def local_css():
    st.markdown("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Montserrat:wght@700;800&family=Poppins:wght@400;600&display=swap');

    header[data-testid="stHeader"] {
        display: none;
    }

    .stApp {
        background-color: #000000;
        padding-top: 0px;
    }

    h1 {
        font-family: 'Montserrat', sans-serif !important;
        text-align: center !important;
        font-weight: 800 !important;
        text-transform: uppercase;
        letter-spacing: 3px;
        margin-top: -30px !important;
        margin-bottom: 20px !important;

        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        background-clip: text;
    }

    [data-testid="stSidebar"] {
        display: none;
    }

    section.main > div {
    background-color: rgba(255, 255, 255, 0.95);
    padding: 3rem 2rem;
    border-radius: 20px;
    box-shadow: 0 20px 40px rgba(0,0,0,0.4);
    max-width: 650px;
    margin: 60px auto;
    }

    div.stButton > button:first-child {
        background-color: transparent;
        color: #764ba2;
        border: 2px solid #764ba2;
        font-weight: 600;
        transition: 0.3s;
    }

    div.stButton > button:first-child:hover {
        background-color: #764ba2;
        color: white;
    }

    .stButton>button[kind="primary"] {
        width: 100%;
        background-color: #764ba2;
        color: white;
        border-radius: 8px;
        height: 3.2em;
        font-family: 'Poppins', sans-serif;
        border: none;
    }

    h3 {
        font-family: 'Montserrat', sans-serif !important;
        font-weight: 700 !important;
        text-align: center;

        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        background-clip: text;
    }

    label {
        font-family: 'Poppins', sans-serif;
        font-weight: 600 !important;
        color: #444 !important;
    }
    </style>
    """, unsafe_allow_html=True)

local_css()

# --- LOAD AI SUMMARIZATION MODEL (CRASH-PROOF METHOD) ---
@st.cache_resource
def load_summarizer():
    print("Loading FLAN-T5 Model safely...")
    tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
    return tokenizer, model

sum_tokenizer, sum_model = load_summarizer()

# --- Helpers ---
def get_relative_time(date_str):
    if not date_str: return "some time ago"
    try:
        past = datetime.datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
        diff = datetime.datetime.utcnow() - past
        days = diff.days
        seconds = diff.seconds
        if days > 365: return f"{days // 365} years ago"
        elif days > 30: return f"{days // 30} months ago"
        elif days > 0: return f"{days} days ago"
        elif seconds > 3600: return f"{seconds // 3600} hours ago"
        elif seconds > 60: return f"{seconds // 60} minutes ago"
        else: return "just now"
    except: return date_str

def is_valid_email(email):
    return re.match(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$", email) is not None

def check_password_strength(password):
    has_upper = bool(re.search(r"[A-Z]", password))
    has_lower = bool(re.search(r"[a-z]", password))
    has_digit = bool(re.search(r"\d", password))
    has_special = bool(re.search(r"[!@#$%^&*(),.?\":{}|<>]", password))
    has_space = bool(re.search(r"\s", password))

    if has_space: return "Weak", ["No spaces allowed"]
    is_alphanum = (has_upper or has_lower) and has_digit

    if len(password) >= 8 and is_alphanum: return "Strong", []
    if len(password) >= 6 and is_alphanum and has_special: return "Medium", ["Add 2 more chars for Strong"]
    if len(password) >= 1: return "Weak", ["Too short (aim for 8+)"]
    return "Weak", ["Enter password"]

# --- Security Logic ---
def generate_otp():
    secret = secrets.token_bytes(20)
    counter = int(time.time())
    msg = struct.pack(">Q", counter)
    hmac_hash = hmac.new(secret, msg, hashlib.sha1).digest()
    offset = hmac_hash[19] & 0xf
    code = ((hmac_hash[offset] & 0x7f) << 24 |
            (hmac_hash[offset + 1] & 0xff) << 16 |
            (hmac_hash[offset + 2] & 0xff) << 8 |
            (hmac_hash[offset + 3] & 0xff))
    otp = code % 1000000
    return f"{otp:06d}"

def create_otp_token(otp, email):
    otp_hash = bcrypt.hashpw(otp.encode('utf-8'), bcrypt.gensalt()).decode('utf-8')
    payload = {
        'otp_hash': otp_hash, 'sub': email, 'type': 'password_reset',
        'iat': datetime.datetime.utcnow(),
        'exp': datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)
    }
    return jwt.encode(payload, SECRET_KEY, algorithm='HS256')

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=['HS256'])
        if payload.get('type') != 'password_reset': return False, "Invalid token type"
        if payload.get('sub') != email: return False, "Token does not belong to this user"
        if bcrypt.checkpw(input_otp.encode('utf-8'), payload['otp_hash'].encode('utf-8')):
            return True, "Valid OTP"
        return False, "Invalid OTP"
    except jwt.ExpiredSignatureError: return False, "OTP Expired"
    except jwt.InvalidTokenError: return False, "Invalid Token"

# --- Email Logic ---
def send_email(to_email, otp, app_pass=None):
    msg = MIMEMultipart()
    msg['From'] = f"PolicyNav <{EMAIL_ADDRESS}>"
    msg['To'] = to_email
    msg['Subject'] = "PolicyNav - Password Reset OTP"
    body = f"<html><body><h3>Your OTP is: {otp}</h3><p>Valid for {OTP_EXPIRY_MINUTES} minutes.</p></body></html>"
    msg.attach(MIMEText(body, 'html'))
    try:
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        password_to_use = app_pass if app_pass else EMAIL_PASSWORD
        if not password_to_use: return False, "No App Password found. Check Secrets."
        server.login(EMAIL_ADDRESS, password_to_use)
        server.sendmail(EMAIL_ADDRESS, to_email, msg.as_string())
        server.quit()
        return True, "Email sent successfully!"
    except Exception as e: return False, str(e)

# --- Visualization Helper ---
def create_gauge(value, title, min_val=0, max_val=100, color="#00ffcc"):
    fig = go.Figure(go.Indicator(
        mode = "gauge+number",
        value = value,
        title = {'text': title, 'font': {'color': color, 'size': 14}},
        number = {'font': {'color': color, 'size': 20}},
        gauge = {
            'axis': {'range': [min_val, max_val], 'tickwidth': 1, 'tickcolor': color},
            'bar': {'color': color},
            'bgcolor': "#1f2937",
            'borderwidth': 2,
            'bordercolor': "#374151",
            'steps': [{'range': [min_val, max_val], 'color': "#0e1117"}],
        }
    ))
    fig.update_layout(paper_bgcolor="#0e1117", font={'color': "#ffffff", 'family': "Courier New"}, height=250, margin=dict(l=10, r=10, t=40, b=10))
    return fig

# --- Navigation & Routing ---
if 'user' not in st.session_state: st.session_state['user'] = None
if 'page' not in st.session_state: st.session_state['page'] = 'login'

def switch_page(page):
    st.session_state['page'] = page
    st.rerun()

def logout():
    st.session_state['user'] = None
    st.session_state['page'] = 'login'
    st.rerun()

def login_page():
    st.title("PolicyNav")
    st.markdown("Login")

    with st.form("login_form"):
        email = st.text_input("Email *")
        password = st.text_input("Password *", type='password')
        submit = st.form_submit_button("Login")

        if submit:
            is_locked, wait_time = db.is_rate_limited(email)
            if is_locked:
                st.error(f"Account Locked! Too many failed attempts. Try again in {int(wait_time)}s.")
            elif not email or not password:
                st.error("Please fill in all mandatory fields (*).")
            elif db.authenticate_user(email, password):
                st.session_state['user'] = email
                st.session_state['page'] = "chat"
                st.balloons()
                st.success(f"Welcome to PolicyNav, {email}!")
                time.sleep(1)
                st.rerun()
            else:
                st.error("Invalid email or password.")
                old_dt = db.check_is_old_password(email, password)
                if old_dt:
                    st.warning(f"You entered an old password from {get_relative_time(old_dt)}. Please use your latest password.")

    st.markdown("---")
    c1, c2 = st.columns(2)
    if c1.button("Create Account"): switch_page("register")
    if c2.button("Forgot Password?"): switch_page("forgot")

def register_page():
    st.title("PolicyNav")
    st.markdown("Create New Account")

    email = st.text_input("Email Address *")
    password = st.text_input("Password *", type='password')

    if password:
        s, f = check_password_strength(password)
        if s == "Weak": st.markdown("Strength: <span class='strength-weak'>Weak</span>", unsafe_allow_html=True)
        elif s == "Medium": st.markdown("Strength: <span class='strength-medium'>Medium</span>", unsafe_allow_html=True)
        else: st.markdown("Strength: <span class='strength-strong'>Strong ✓</span>", unsafe_allow_html=True)
        if f: st.caption(f"Issues: {', '.join(f)}")

    if st.button("Register"):
        if not email or not password:
            st.error("Please fill in all mandatory fields (*).")
        elif not is_valid_email(email):
            st.error("Invalid email format.")
        else:
            strength, feedback = check_password_strength(password)
            if strength == "Weak":
                st.error(f"Password is too weak: {', '.join(feedback)}")
            elif db.register_user(email, password):
                st.success("Registration Successful! Redirecting to login...")
                time.sleep(2)
                switch_page("login")
            else:
                st.error("User with this email already exists.")

    st.markdown("---")
    if st.button("Return to Login"): switch_page("login")

def forgot_page():
    st.title("PolicyNav")
    st.markdown("Password Recovery")

    if 'stage' not in st.session_state: st.session_state['stage'] = 'email'

    if st.session_state['stage'] == 'email':
        email = st.text_input("Enter your registered Email *")
        if st.button("Next"):
            if not email: st.error("Email is mandatory (*).")
            elif not is_valid_email(email): st.error("Invalid email format.")
            elif db.check_user_exists(email):
                st.session_state['reset_email'] = email
                st.session_state['stage'] = 'otp'
                st.rerun()
            else: st.error("Email not found in our database.")
        st.markdown("---")
        if st.button("Return to Login"): switch_page("login")

    elif st.session_state['stage'] == 'otp':
        st.info(f"Account found: {st.session_state['reset_email']}")
        app_pass = EMAIL_PASSWORD if EMAIL_PASSWORD else st.text_input("Enter Google App Password manually *", type="password")

        if st.button("Send Verification Code"):
            if app_pass:
                otp = generate_otp()
                with st.spinner("Sending OTP..."):
                    success, msg = send_email(st.session_state['reset_email'], otp, app_pass)
                if success:
                    st.session_state['token'] = create_otp_token(otp, st.session_state['reset_email'])
                    st.session_state['stage'] = 'verify'
                    st.success("OTP Sent!")
                    time.sleep(1)
                    st.rerun()
                else: st.error(f"Failed to send email: {msg}")
            else: st.error("App Password is required (*).")
        st.markdown("---")
        if st.button("Cancel"):
            st.session_state['stage'] = 'email'
            st.rerun()

    elif st.session_state['stage'] == 'verify':
        st.info("Check your email for the code.")
        otp_input = st.text_input("Enter 6-digit OTP *", max_chars=6)
        if st.button("Verify OTP"):
            if not otp_input: st.error("OTP is required (*)")
            else:
                valid, msg = verify_otp_token(st.session_state['token'], otp_input, st.session_state['reset_email'])
                if valid:
                    st.session_state['stage'] = 'reset'
                    st.success("Verified!")
                    time.sleep(1)
                    st.rerun()
                else: st.error(msg)
        if st.button("Resend Code"):
            st.session_state['stage'] = 'otp'
            st.rerun()

    elif st.session_state['stage'] == 'reset':
        p1 = st.text_input("New Password *", type='password')
        p2 = st.text_input("Confirm New Password *", type='password')
        if st.button("Update Password"):
            if not p1 or not p2: st.error("All password fields are mandatory (*).")
            elif p1 != p2: st.error("Passwords do not match.")
            elif db.check_password_reused(st.session_state['reset_email'], p1): st.error("Old password reuse is not permitted.")
            else:
                strength, _ = check_password_strength(p1)
                if strength == "Weak": st.error("Password is too weak.")
                else:
                    db.update_password(st.session_state['reset_email'], p1)
                    st.balloons()
                    st.success("Password Updated! Please Login.")
                    for key in ['stage', 'reset_email', 'token']:
                        if key in st.session_state: del st.session_state[key]
                    time.sleep(2)
                    switch_page("login")
    if st.button("Cancel Recovery"): switch_page("login")

from rag_engine import generate_answer

def chat_page():
    if not st.session_state['user']: switch_page('login'); return
    st.title("PolicyNav Chat")

    col1, col2 = st.columns(2)
    languages = {"English": "en", "Hindi": "hi", "Telugu": "te", "Tamil": "ta", "Kannada": "kn", "Malayalam": "ml", "Marathi": "mr", "Bengali": "bn", "Gujarati": "gu", "Punjabi": "pa"}
    with col1: q_lang_name = st.selectbox("Question Language", list(languages.keys()))
    with col2: ans_lang_name = st.selectbox("Answer Language", list(languages.keys()))

    if "messages" not in st.session_state: st.session_state.messages = []
    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]): st.markdown(msg["content"])

    if prompt := st.chat_input("Ask about government policies..."):
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"): st.markdown(prompt)
        with st.chat_message("assistant"):
            with st.spinner("Searching policy database..."):
                response = generate_answer(prompt, languages[q_lang_name], languages[ans_lang_name])
            st.markdown(response)
        st.session_state.messages.append({"role": "assistant", "content": response})

def knowledge_graph_page():
    if not st.session_state['user']: switch_page('login'); return
    st.title("Policy Knowledge Map")
    try:
        path = "/content/drive/MyDrive/PolicyNav/chunks.pkl"
        if os.path.exists(path):
            with open(path, "rb") as f: chunks = pickle.load(f)
            with st.spinner("Mapping relationships..."):
                kg = PolicyKG()
                kg.add_policy_data(chunks)
                html_binary = kg.visualize_kg()
                components.html(html_binary, height=600)
        else: st.error("Vector data not found. Run processing first.")
    except Exception as e: st.error(f"Error loading graph: {e}")

def readability_page():
    if not st.session_state['user']: switch_page('login'); return
    st.title("Text Readability Analyzer")
    tab1, tab2 = st.tabs(["Input Text", "Upload File"])
    text_input = ""
    with tab1:
        raw_text = st.text_area("Enter text to analyze (min 50 chars):", height=200)
        if raw_text: text_input = raw_text
    with tab2:
        uploaded_file = st.file_uploader("Upload a file", type=["txt", "pdf"])
        if uploaded_file:
            if uploaded_file.type == "application/pdf":
                reader = PyPDF2.PdfReader(uploaded_file)
                text_input = "".join([page.extract_text() + "\n" for page in reader.pages])
            else: text_input = uploaded_file.read().decode("utf-8")

    if st.button("Analyze Readability", type="primary"):
        if len(text_input) < 50: st.error("Text is too short.")
        else:
            import readability
            with st.spinner("Calculating advanced metrics..."):
                analyzer = readability.ReadabilityAnalyzer(text_input)
                score = analyzer.get_all_metrics()
            st.markdown("---")
            c1, c2, c3 = st.columns(3)
            with c1: st.plotly_chart(create_gauge(score["Flesch Reading Ease"], "Flesch Ease", 0, 100), use_container_width=True)
            with c2: st.plotly_chart(create_gauge(score["Flesch-Kincaid Grade"], "Grade Level", 0, 20), use_container_width=True)
            with c3: st.plotly_chart(create_gauge(score["SMOG Index"], "SMOG Index", 0, 20), use_container_width=True)

# --- YOUR NEW SUMMARIZATION PAGE ---
def summarization_page():
    if not st.session_state['user']: switch_page('login'); return
    st.title("AI Policy Summarizer")
    st.markdown("Condense long, complex policy texts into quick, multilingual bullet points.")

    languages = {"English": "en", "Hindi": "hi", "Telugu": "te", "Tamil": "ta", "Kannada": "kn", "Marathi": "mr", "Bengali": "bn"}
    target_lang_name = st.selectbox("Select Output Language", list(languages.keys()))

    text_to_summarize = st.text_area("Paste policy document text here:", height=250)

    if st.button("Generate Summary", type="primary"):
        if len(text_to_summarize.strip()) < 50:
            st.error("Please enter at least 50 characters of text to summarize.")
        else:
            with st.spinner(f"Extracting key points and translating to {target_lang_name}..."):
                try:
                    # 1. Generate English Summary using the manual crash-proof method
                    prompt = f"Summarize the following text into 3 distinct key bullet points:\n{text_to_summarize[:2000]}"
                    inputs = sum_tokenizer(prompt, return_tensors="pt", truncation=True)
                    outputs = sum_model.generate(**inputs, max_new_tokens=150)
                    summary_result = sum_tokenizer.decode(outputs[0], skip_special_tokens=True)

                    # 2. Translate using teammate's deep-translator implementation
                    target_code = languages[target_lang_name]
                    if target_code != 'en':
                        final_summary = GoogleTranslator(source='en', target=target_code).translate(summary_result)
                    else:
                        final_summary = summary_result

                    # 3. Display Result beautifully
                    st.success("Summary Generated Successfully!")
                    st.markdown(f"""
                    <div style="background-color: #1f2937; padding: 20px; border-radius: 10px; border-left: 5px solid #764ba2;">
                        <h4 style="margin:0 0 10px 0; color: #ffffff !important;">📝 Final Summary</h4>
                        <p style="margin:0; color: #e5e7eb; font-size: 1.1rem; line-height: 1.6;">{final_summary}</p>
                    </div>
                    """, unsafe_allow_html=True)
                except Exception as e:
                    st.error(f"Failed to generate summary: {e}")

def admin_page():
    if st.session_state['user'] != "admin@llm.com": st.error("Access Denied"); return
    st.title("Admin Panel")
    users = db.get_all_users()
    st.metric("Total Users", len(users))
    st.markdown("---")
    for u_email, u_created in users:
        c1, c2, c3 = st.columns([3, 2, 1])
        c1.write(f"{u_email}"); c2.write(u_created)
        if u_email != "admin@llm.com":
            if c3.button("Delete", key=u_email, type="primary"):
                db.delete_user(u_email); st.warning(f"Deleted {u_email}"); time.sleep(0.5); st.rerun()

# --- Updated Main Router with 6 Columns ---
if st.session_state['user']:

    c1, c2, c3, c4, c5, c6 = st.columns(6)

    with c1:
        if st.button("Chat"): switch_page("chat")
    with c2:
        if st.button("Readability"): switch_page("readability")
    with c3:
        if st.button("Graph"): switch_page("graph")
    with c4:
        if st.button("Summary"): switch_page("summarization") # <- Your New Button!

    if st.session_state['user'] == "admin@llm.com":
        with c5:
            if st.button("Admin"): switch_page("admin")
        with c6:
            if st.button("Log Out"): logout()
    else:
        with c5:
            if st.button("Log Out"): logout()

    # Routing rules
    if st.session_state["page"] == "chat": chat_page()
    elif st.session_state["page"] == "readability": readability_page()
    elif st.session_state["page"] == "graph": knowledge_graph_page()
    elif st.session_state["page"] == "summarization": summarization_page() # <- Your New Page Route!
    elif st.session_state["page"] == "admin": admin_page()
else:
    if st.session_state["page"] == "login": login_page()
    elif st.session_state["page"] == "register": register_page()
    elif st.session_state["page"] == "forgot": forgot_page()

Writing app.py


In [ ]:
import os
import subprocess
import time
from google.colab import userdata
from pyngrok import ngrok

# 1. Retrieve secrets safely
email_pass = None; ngrok_token = None
try:
    try:
        email_pass = userdata.get('EMAIL_APP_PASSWORD')
    except Exception as e:
        print(f"Warning: EMAIL_PASSWORD secret not found: {e}")

    try:
        ngrok_token = userdata.get('NGROK_AUTHTOKEN')
    except Exception as e:
        print(f"Warning: NGROK_AUTHTOKEN secret not found: {e}")

    try:
        jwt_secret = userdata.get('JWT_SECRET_KEY')
    except Exception as e:
        print(f"Warning: JWT_SECRET_KEY secret not found: {e}")

    # Store in environment for the subprocess to see
    if email_pass:
        os.environ['EMAIL_APP_PASSWORD'] = email_pass
    if jwt_secret:
        os.environ['JWT_SECRET_KEY'] = jwt_secret

except Exception as e:
    print(f"Error setting up environment: {e}")



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import sqlite3

base_dir = "/content/drive/MyDrive/PolicyNav/documents"
db_path = "/content/drive/MyDrive/PolicyNav/policies_meta.db"

# Create folder
os.makedirs(base_dir, exist_ok=True)

# Create database + table
conn = sqlite3.connect(db_path)

c = conn.cursor()

c.execute("""
CREATE TABLE IF NOT EXISTS policies (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    url TEXT,
    local_path TEXT
)
""")

conn.commit()
conn.close()

print("Database initialized successfully!")
print("PDF folder:", base_dir)
print("Database:", db_path)

Database initialized successfully!
PDF folder: /content/drive/MyDrive/PolicyNav/documents
Database: /content/drive/MyDrive/PolicyNav/policies_meta.db


In [ ]:
# import requests
# from bs4 import BeautifulSoup
# from urllib.parse import urljoin
# from tqdm import tqdm
# import urllib3
# import time
# import os
# import sqlite3

# urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# # Paths
# base_dir = "/content/drive/MyDrive/PolicyNav/documents"
# db_path = "/content/drive/MyDrive/PolicyNav/policies_meta.db"

# os.makedirs(base_dir, exist_ok=True)

# HEADERS = {
#     "User-Agent": "Mozilla/5.0"
# }

# # -----------------------------
# # Known Official Policy PDFs
# # -----------------------------
# direct_pdfs = [

# ("NEP","https://www.education.gov.in/sites/upload_files/mhrd/files/NEP_Final_English_0.pdf"),
# ("PMJDY","https://www.pmjdy.gov.in/files/E-Documents/PMJDY_Brochure_ENG.pdf"),
# ("PMAY_Urban","https://pmay-urban.gov.in/uploads/guidelines/Operational-Guidelines-of-PMAY-U.pdf"),
# ("PMAY_Gramin","https://pmayg.dord.gov.in/netiayHome/Uploaded/Guidelines-English_Book_Final.pdf"),
# ("PMFBY","https://pmfby.gov.in/pdf/Revamped%20Operational%20Guidelines_17th%20August%202020.pdf"),
# ("PMKISAN","https://pmkisan.gov.in/Documents/RevisedPM-KISANOperationalGuidelines(English).pdf"),
# ("JalJeevanMission","https://jaljeevanmission.gov.in/sites/default/files/guideline/JJM_Operational_Guidelines.pdf"),
# ("SHC","https://www.agriwelfare.gov.in/sites/default/files/Guidelines_for_Demonstrations_under_SHC_Scheme.pdf"),
# ("SwachhBharat","https://archive.ids.ac.uk/clts/sites/communityledtotalsanitation.org/files/SwachhBharatMissionGraminGuidelines.pdf"),
# ("PMKVY","https://www.msde.gov.in/static/uploads/2024/02/PMKVY-4.0-Guidelines_final-copy.pdf"),
# ("DigitalIndia","https://www.meity.gov.in/static/uploads/2024/03/Brochure.pdf"),
# ("SmartCities","https://mohua.gov.in/dataSmartCities/uploads/resource/resourceDoc/Resource_Doc_1723187622_Making_a_City_Smart_Learnings_from_the_Smart_Cities_Mission.pdf")

# ]


# # -----------------------------
# # Policy Landing Pages
# # -----------------------------
# policy_pages = {
# "PMJDY":"https://pmjdy.gov.in/",
# "PMAY_Urban":"https://pmay-urban.gov.in/",
# "JalJeevanMission":"https://jaljeevanmission.gov.in/",
# "DigitalIndia":"https://digitalindia.gov.in/",
# "SkillIndia":"https://www.msde.gov.in/"
# }


# # -----------------------------
# # Find PDFs from websites
# # -----------------------------
# def find_pdfs(url):

#     try:
#         r = requests.get(url,headers=HEADERS,timeout=20,verify=False)

#         soup = BeautifulSoup(r.text,"html.parser")

#         pdfs = []

#         for link in soup.find_all("a",href=True):

#             if ".pdf" in link["href"].lower():

#                 pdfs.append(urljoin(url,link["href"]))

#         return pdfs

#     except:

#         return []


# # -----------------------------
# # Check already downloaded
# # -----------------------------
# def already_downloaded(url):

#     conn = sqlite3.connect(db_path)
#     c = conn.cursor()

#     c.execute("SELECT id FROM policies WHERE url=?",(url,))
#     data = c.fetchone()

#     conn.close()

#     return data is not None


# # -----------------------------
# # Download PDF
# # -----------------------------
# def download_pdf(url,scheme):

#     filename = url.split("/")[-1].split("?")[0]

#     save_name = f"{scheme}_{filename}"

#     path = os.path.join(base_dir,save_name)

#     if os.path.exists(path) or already_downloaded(url):
#         return

#     try:

#         r = requests.get(url,headers=HEADERS,stream=True,timeout=30,verify=False)

#         if "application/pdf" not in r.headers.get("Content-Type","").lower() and not r.content.startswith(b"%PDF"):
#             return

#         with open(path,"wb") as f:

#             for chunk in r.iter_content(1024):

#                 f.write(chunk)

#         conn = sqlite3.connect(db_path)
#         c = conn.cursor()

#         c.execute(
#             "INSERT INTO policies (title,url,local_path) VALUES (?,?,?)",
#             (scheme,url,path)
#         )

#         conn.commit()
#         conn.close()

#         print("Downloaded:",save_name)

#     except:

#         print("Failed:",url)


# # -----------------------------
# # Combine sources
# # -----------------------------
# pdf_mapping = []

# # Add direct policy PDFs
# for scheme,url in direct_pdfs:
#     pdf_mapping.append((url,scheme))

# # Scrape websites
# print("Scanning policy websites...")

# for scheme,url in policy_pages.items():

#     pdfs = find_pdfs(url)

#     print(f"{scheme}: {len(pdfs)} PDFs found")

#     for pdf in pdfs:
#         pdf_mapping.append((pdf,scheme))


# print("\nTotal PDFs to download:",len(pdf_mapping))
# print("\nStarting downloads...\n")


# for pdf_url,scheme in pdf_mapping:

#     download_pdf(pdf_url,scheme)

#     time.sleep(1)


# # -----------------------------
# # Final Count
# # -----------------------------
# conn = sqlite3.connect(db_path)
# c = conn.cursor()

# c.execute("SELECT COUNT(*) FROM policies")
# count = c.fetchone()[0]

# conn.close()

# print("\nDownload complete!")
# print("Total PDFs stored in database:",count)
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin,urlparse
from tqdm import tqdm
import urllib3
import time
import os
import sqlite3
import re

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ==============================
# PATHS
# ==============================

base_dir = "/content/drive/MyDrive/PolicyNav/documents"
db_path = "/content/drive/MyDrive/PolicyNav/policies_meta.db"

os.makedirs(base_dir, exist_ok=True)

HEADERS = {
"User-Agent":
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36"
}

# ==============================
# DATABASE
# ==============================

conn = sqlite3.connect(db_path)
c = conn.cursor()

c.execute("""
CREATE TABLE IF NOT EXISTS policies(
id INTEGER PRIMARY KEY AUTOINCREMENT,
title TEXT,
url TEXT UNIQUE,
local_path TEXT
)
""")

conn.commit()
conn.close()

# ==============================
# LARGE GOVT SOURCES
# ==============================

policy_pages = {

"IndiaGov":"https://www.india.gov.in/",
"NITI":"https://www.niti.gov.in/",
"PRS":"https://prsindia.org/",
"RBI":"https://www.rbi.org.in/",
"DigitalIndia":"https://digitalindia.gov.in/",
"JalJeevan":"https://jaljeevanmission.gov.in/",
"PMAY":"https://pmay-urban.gov.in/",
"SkillIndia":"https://www.msde.gov.in/",
"Agriculture":"https://agriwelfare.gov.in/",
"Health":"https://main.mohfw.gov.in/",
"Education":"https://www.education.gov.in/",
"Environment":"https://moef.gov.in/",
"Finance":"https://dea.gov.in/",
"LawMinistry":"https://lawmin.gov.in/",
"Parliament":"https://sansad.in/",
"UIDAI":"https://uidai.gov.in/",
"MeitY":"https://www.meity.gov.in/",
"RuralDev":"https://rural.nic.in/",
"Water":"https://jalshakti-dowr.gov.in/",
"PIB":"https://pib.gov.in/",
"DataGov":"https://data.gov.in/",
}

# ==============================
# SAFE REQUEST
# ==============================

def safe_request(url):

    for _ in range(3):

        try:

            r = requests.get(
                url,
                headers=HEADERS,
                timeout=30,
                verify=False,
                allow_redirects=True
            )

            return r

        except:

            time.sleep(2)

    return None


# ==============================
# CLEAN FILE NAME
# ==============================

def clean_filename(name):

    name = re.sub(r'[<>:"/\\|?*]','_',name)

    return name[:200]


# ==============================
# FIND PDF LINKS
# ==============================

def extract_links(base_url):

    r = safe_request(base_url)

    if not r:
        return [],[]

    soup = BeautifulSoup(r.text,"html.parser")

    pdfs = []
    pages = []

    for link in soup.find_all("a",href=True):

        href = link["href"]

        full = urljoin(base_url,href)

        if ".pdf" in href.lower():

            pdfs.append(full)

        else:

            if base_url in full:
                pages.append(full)

    return pdfs,pages


# ==============================
# CHECK DB
# ==============================

def already_downloaded(url):

    conn = sqlite3.connect(db_path)
    c = conn.cursor()

    c.execute("SELECT id FROM policies WHERE url=?",(url,))
    data = c.fetchone()

    conn.close()

    return data is not None


# ==============================
# DOWNLOAD PDF
# ==============================

def download_pdf(url,source):

    filename = url.split("/")[-1].split("?")[0]

    filename = clean_filename(filename)

    save_name = f"{source}_{filename}"

    path = os.path.join(base_dir,save_name)

    if os.path.exists(path) or already_downloaded(url):

        return

    r = safe_request(url)

    if not r:
        print("Failed:",url)
        return

    try:

        if "pdf" not in r.headers.get("Content-Type","").lower() and not r.content.startswith(b"%PDF"):

            return

        with open(path,"wb") as f:

            f.write(r.content)

        conn = sqlite3.connect(db_path)
        c = conn.cursor()

        c.execute(
            "INSERT INTO policies (title,url,local_path) VALUES (?,?,?)",
            (source,url,path)
        )

        conn.commit()
        conn.close()

        print("Downloaded:",save_name)

    except:

        print("Failed:",url)


# ==============================
# CRAWL WEBSITE (2 LEVEL)
# ==============================

pdf_links = set()

print("Scanning government portals...\n")

for name,url in policy_pages.items():

    print("Scanning:",name)

    pdfs,pages = extract_links(url)

    pdf_links.update(pdfs)

    # crawl subpages
    for page in pages[:20]:

        sub_pdfs,_ = extract_links(page)

        pdf_links.update(sub_pdfs)


print("\nTotal PDFs discovered:",len(pdf_links))


# ==============================
# DOWNLOAD LOOP
# ==============================

print("\nStarting downloads...\n")

for pdf in tqdm(list(pdf_links)):

    domain = urlparse(pdf).netloc.replace("www.","")

    download_pdf(pdf,domain)

    time.sleep(0.3)


# ==============================
# FINAL COUNT
# ==============================

conn = sqlite3.connect(db_path)
c = conn.cursor()

c.execute("SELECT COUNT(*) FROM policies")

count = c.fetchone()[0]

conn.close()

print("\nDownload complete!")
print("Total PDFs stored:",count)

Scanning government portals...

Scanning: IndiaGov
Scanning: NITI
Scanning: PRS
Scanning: RBI
Scanning: DigitalIndia
Scanning: JalJeevan
Scanning: PMAY
Scanning: SkillIndia
Scanning: Agriculture
Scanning: Health
Scanning: Education
Scanning: Environment
Scanning: Finance
Scanning: LawMinistry
Scanning: Parliament
Scanning: UIDAI
Scanning: MeitY
Scanning: RuralDev
Scanning: Water
Scanning: PIB
Scanning: DataGov


/tmp/ipykernel_970/354953164.py:310: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(r.text,"html.parser")



Total PDFs discovered: 3633

Starting downloads...



  0%|          | 0/3633 [00:00<?, ?it/s]

Downloaded: jaljeevanmission.gov.in_FHTC_Gurdaspur.pdf


...

 19%|█▊        | 681/3633 [52:37<4:26:46,  5.42s/it]

Downloaded: jaljeevanmission.gov.in_FHTC_Firozabad.pdf


 19%|█▉        | 682/3633 [52:40<3:51:56,  4.72s/it]

Downloaded: pmay-urban.gov.in_WtxPolaUfZjJ62C9SBI_3000000000_06-03-2020.pdf


 19%|█▉        | 683/3633 [52:47<4:24:34,  5.38s/it]

In [ ]:
import sqlite3

db_path = "/content/drive/MyDrive/PolicyNav/policies_meta.db"

conn = sqlite3.connect(db_path)
c = conn.cursor()

c.execute("SELECT COUNT(*) FROM policies")
print("Total PDFs stored:", c.fetchone()[0])

conn.close()

In [ ]:
from pypdf import PdfReader
import os

base_dir = "/content/drive/MyDrive/PolicyNav/documents"

documents = []

for file in os.listdir(base_dir):
    if file.endswith(".pdf"):
        path = os.path.join(base_dir, file)
        try:
            reader = PdfReader(path)
            text = ""
            for page in reader.pages:
                if page.extract_text():
                    text += page.extract_text()

            documents.append({
                "file": file,
                "text": text
            })
        except:
            print("Error reading:", file)

print("Total PDFs processed:", len(documents))

In [ ]:
def split_text(text, chunk_size=500):

    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

all_chunks = []
for doc in documents:
    chunks = split_text(doc["text"])
    for chunk in chunks:
        all_chunks.append({
            "source": doc["file"],
            "text": chunk
        })
print("Total chunks created:", len(all_chunks))

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [chunk["text"] for chunk in all_chunks]
embeddings = model.encode(texts, show_progress_bar=True)
print("Embedding shape:", embeddings.shape)

In [ ]:
import faiss
import numpy as np
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))
print("Vectors stored:", index.ntotal)

In [ ]:
faiss.write_index(index, "/content/drive/MyDrive/PolicyNav/policy_vector_db.index")
import pickle
with open("/content/drive/MyDrive/PolicyNav/chunks.pkl","wb") as f:
    pickle.dump(all_chunks,f)
print("Vector database saved!")

In [ ]:
%%writefile rag_engine.py
import pickle
import faiss
from sentence_transformers import SentenceTransformer
from deep_translator import GoogleTranslator

# LOAD EMBEDDING MODEL
model = SentenceTransformer("all-MiniLM-L6-v2")

# LOAD VECTOR DATABASE
index = faiss.read_index("/content/drive/MyDrive/PolicyNav/policy_vector_db.index")

# LOAD DOCUMENT CHUNKS
with open("/content/drive/MyDrive/PolicyNav/chunks.pkl", "rb") as f:
    chunks = pickle.load(f)


# ---------- TRANSLATION FUNCTION ----------
def translate_text(text, source_lang, target_lang):

    if source_lang == target_lang:
        return text

    try:
        translated = GoogleTranslator(
            source=source_lang,
            target=target_lang
        ).translate(text)

        return translated

    except:
        return text


# ---------- SEARCH ----------
def search_policy(query, top_k=5):

    query_vector = model.encode([query])

    distances, indices = index.search(query_vector, top_k)

    results = []
    for i in indices[0]:
        results.append(chunks[i]["text"])

    return results


# ---------- MAIN RAG ----------
def generate_answer(query, q_lang="en", ans_lang="en"):

    # Translate question to English
    query_en = translate_text(query, q_lang, "en")

    # Retrieve context
    context_chunks = search_policy(query_en)

    context = "\n".join(context_chunks)

    # Limit context size to avoid translator failure
    context = context[:1500]

    answer_en = f"""
Based on policy documents:

{context}

Question: {query_en}
"""

    # Translate answer
    final_answer = translate_text(answer_en, "en", ans_lang)

    return final_answer

In [ ]:
import os
import subprocess
import time
from pyngrok import ngrok

# --- STEP 1: Paste your token directly here ---
MY_DIRECT_TOKEN = "39Ynjm9MxySrlGhNowfEQGXwjuY_7fdXJo6PSN8vaYG8WP9Vq"

# --- STEP 2: Authenticate and Run ---
ngrok.set_auth_token(MY_DIRECT_TOKEN)
ngrok.kill() # Clears any old sessions
time.sleep(1)

# 3. Run Streamlit
# We still use os.environ.copy() so the app can see your Drive paths
print("⏳ Starting Streamlit...")
process = subprocess.Popen(['streamlit', 'run', 'app.py'], env=os.environ.copy())
time.sleep(5)

# 4. Open Tunnel
try:
    public_url = ngrok.connect(8501).public_url
    print(f"\n🚀 App Running: {public_url}")
    print("\n✨ Click the link above to see your app!")
except Exception as e:
    print(f"❌ Ngrok Error: {e}")

# 5. Keep Alive
try:
    input("\n🛑 Press ENTER here to STOP the server...\n")
except:
    pass
finally:
    process.terminate()
    ngrok.kill()
    print("✅ Stopped.")